In [1]:
import xarray as xr

ds = xr.open_dataset("/Users/rafaela.dsp/Downloads/epoch_data.nc")
print(ds)

sujets = ["sub-006", "sub-008", "sub-009", "sub-013", "sub-014", "sub-015"]
noms = [s + "_task-categorisation" for s in sujets]
petit = ds.sel(recording=noms).load()
petit.to_netcdf("/Users/rafaela.dsp/Desktop/EPFL/MA1/Machine Learning/ML_EEG_FoodImages/epoch_data_6sujets.nc")
ds.close()
print("Fichier réduit créé")

<xarray.Dataset> Size: 15GB
Dimensions:            (recording: 112, epoch: 721, channel: 64, sample: 705)
Coordinates: (12/26)
  * epoch              (epoch) int64 6kB 0 1 2 3 4 5 ... 715 716 717 718 719 720
  * channel            (channel) <U3 768B 'Fp1' 'AF7' 'AF3' ... 'PO8' 'PO4' 'O2'
  * sample             (sample) int64 6kB -64 -63 -62 -61 ... 637 638 639 640
    event_name         (recording, epoch) <U7 2MB ...
    stimulus_time      (recording, epoch) float64 646kB ...
    response_time      (recording, epoch) float64 646kB ...
    ...                 ...
    choice_rt          (recording, epoch) float64 646kB ...
    resp_too_slow      (recording, epoch) float64 646kB ...
    sample_continuous  (recording, epoch) float64 646kB ...
  * recording          (recording) <U27 12kB 'sub-006_task-categorisation' .....
    subject            (recording) <U3 1kB ...
    task               (recording) <U14 6kB ...
Data variables:
    data               (recording, epoch, channel, sample) 

In [2]:
xr.load_dataset("/Users/rafaela.dsp/Desktop/EPFL/MA1/Machine Learning/ML_EEG_FoodImages/epoch_data_6sujets.nc")
print(petit.recording.values)

['sub-006_task-categorisation' 'sub-008_task-categorisation'
 'sub-009_task-categorisation' 'sub-013_task-categorisation'
 'sub-014_task-categorisation' 'sub-015_task-categorisation']


In [3]:
import numpy as np

s = petit.sel(recording="sub-006_task-categorisation")
m = s.event_name.values == "display"

print(np.unique(s.event_type.values[m], return_counts=True))
print(np.unique(s.resp_choice.values[m], return_counts=True))
print(s.food_image_code.values[m][:10])

(array(['decision', 'health', 'taste'], dtype='<U8'), array([120, 120, 120]))
(array(['No', 'Yes'], dtype='<U3'), array([148, 212]))
['foodImage.46' 'foodImage.49' 'foodImage.114' 'foodImage.54'
 'foodImage.98' 'foodImage.61' 'foodImage.79' 'foodImage.10'
 'foodImage.45' 'foodImage.19']


In [5]:
sub6 = petit.sel(recording="sub-006_task-categorisation")
events = sub6.event_name.values

masque = events == "display"

eeg_data = sub6['data'].values[masque]

print("Shape of EEG data for 'display' events:", eeg_data.shape)

Shape of EEG data for 'display' events: (360, 64, 705)


### First try Healthy

In [19]:
event_types = sub6.event_type.values
masque_health = (events == "display") & (event_types == "health")

eeg_data_health = sub6['data'].values[masque_health]
print("Shape of EEG data for 'display' events with 'health' event type:", eeg_data_health.shape)

choices = sub6.resp_choice.values[masque_health]
print("Unique response choices for 'display' events with 'health' event type:", np.unique(choices, return_counts=True))

choices_0_1 = (choices == "Yes").astype(int)

time = sub6.sample.values
time = time/256

masque_time = (time >= 0.2) & (time <= 0.5)

eeg_data_time = eeg_data_health[:,:,masque_time]
mean_time = eeg_data_time.mean(axis=2)

print(mean_time.shape)
    


Shape of EEG data for 'display' events with 'health' event type: (120, 64, 705)
Unique response choices for 'display' events with 'health' event type: (array(['No', 'Yes'], dtype='<U3'), array([62, 58]))
(120, 64)
